### Job shop scheduling problem

The job shop scheduling problem consists of minimizing the total time required to complete a production plan involving a given set of jobs and machines. This quantity is known as the **makespan**. The problem is of the min-max type, since it effectively minimizes the maximum completion time of the production schedule.

* $J$: Set of jobs
* $M$: Set of machines
* $O_j$: Set of operations of job $j \in J$, in sequential order
* $p_{j,o}$: Processing time of operation $o$ of job $j$
* $m_{j,o}$: Machine assigned to operation $o$ of job $j$
* $Big_M$: A sufficiently large constant

* $S_{j,o} \geq 0$: Start time of operation $o$ of job $j$
* $C_{\max} \geq 0$: Total completion time (makespan)
* $y_{(j,o),(j',o')} \in \{0,1\}$: Binary variable indicating the execution order between two operations that require the same machine

$$
\begin{aligned}
\min \quad & C_{\max} \\
\textbf{s.t.} \quad
& S_{j,o+1} \geq S_{j,o} + p_{j,o} && \forall j \in J,\ o < |O_j| \\
& S_{j,o} + p_{j,o} \leq S_{j',o'} + Big_M(1 - y_{(j,o),(j',o')}) \\
& S_{j',o'} + p_{j',o'} \leq S_{j,o} + Big_M y_{(j,o),(j',o')} \\
& \forall (j,o),(j',o'): m_{j,o} = m_{j',o'},\ (j,o) \neq (j',o') \\
& C_{\max} \geq S_{j,|O_j|-1} + p_{j,|O_j|-1} && \forall j \in J
\end{aligned}
$$


In [ ]:
%pip install -q amplpy numpy matplotlib pandas networkx folium
from amplpy import AMPL, ampl_notebook
import numpy as np

# HiGHS is the default. Gurobi requires an AMPL-compatible license.
SOLVER = "highs"  # or "gurobi"
LICENSE_UUID = "default"  # Colab Community Edition; use your UUID locally
runtime = ampl_notebook(modules=[SOLVER], license_uuid=LICENSE_UUID)

def new_ampl():
    return AMPL()

def solve_checked(model):
    model.solve(solver=SOLVER)
    if model.solve_result != "solved":
        raise RuntimeError(f"No proven optimal solution: {model.solve_result}. "
                           "Inspect the solver log before extracting values.")

def values(model, name):
    # Numeric dictionaries keep plotting independent of the solver API.
    return model.var[name].get_values().to_dict()




In [ ]:
# Data
jobs = {
    "J1": [("M1", 3), ("M2", 2), ("M3", 2)],
    "J2": [("M2", 2), ("M1", 1), ("M3", 4)],
    "J3": [("M3", 4), ("M1", 3), ("M2", 1)]
}
machines = {"M1", "M2", "M3"}

# Sets
operations = []  # (job, operation number, machine)
for j, ops in jobs.items():
    for i, (m, d) in enumerate(ops):
        operations.append((j, i, m))

operations = [(j,i,m) for j,ops in jobs.items() for i,(m,_) in enumerate(ops)]
durations = {(j,i,m): jobs[j][i][1] for j,i,m in operations}
precedence = [(j,i,m,j,i+1,jobs[j][i+1][0])
              for j,i,m in operations if i+1 < len(jobs[j])]
conflicts = [a+b for idx,a in enumerate(operations) for b in operations[idx+1:] if a[2]==b[2]]
model = new_ampl()
model.eval(r"""
set O dimen 3;
set P dimen 6;
set E dimen 6;
param duration {O} > 0;
param M > 0;
var start {O} >= 0;
var Cmax >= 0;
var before {E} binary;
minimize Total_Cost: Cmax;
subject to Precedence {(j,i,m,j2,i2,m2) in P}:
    start[j2,i2,m2] >= start[j,i,m] + duration[j,i,m];
subject to FirstBeforeSecond {(j,i,m,j2,i2,m2) in E}:
    start[j,i,m] + duration[j,i,m] <= start[j2,i2,m2] + M*(1-before[j,i,m,j2,i2,m2]);
subject to SecondBeforeFirst {(j,i,m,j2,i2,m2) in E}:
    start[j2,i2,m2] + duration[j2,i2,m2] <= start[j,i,m] + M*before[j,i,m,j2,i2,m2];
subject to Completion {(j,i,m) in O}: Cmax >= start[j,i,m] + duration[j,i,m];
""")
model.set["O"] = operations
model.set["P"] = precedence
model.set["E"] = conflicts
model.param["duration"] = durations
model.param["M"] = 1000  # Same bound as the original teaching formulation

solve_checked(model)
start = values(model, "start")
Cmax = model.var["Cmax"].value()
for operation, time in start.items():
    print(f"start{operation} = {time:.1f}")
print(f"Makespan = {Cmax:.1f}")


In [ ]:
import matplotlib.pyplot as plt
# Make data available for Gantt diagram
gantt_data = []
for (j, i, m) in operations:
    s = start[j, i, m]
    d = jobs[j][i][1]
    gantt_data.append((m, s, d, j, i))

# Plot Gantt diagram
fig, ax = plt.subplots(figsize=(10, 4))
colors = {'J1': 'tab:blue', 'J2': 'tab:green', 'J3': 'tab:orange'}
for m_idx, (m, s, d, j, i) in enumerate(sorted(gantt_data, key=lambda x: x[0])):
    ax.barh(m, d, left=s, color=colors[j])
    ax.text(s + d/2, m, f"{j}-{i}", ha='center', va='center', color='white')

ax.set_xlabel("Time")
ax.set_ylabel("Machine")
ax.set_title("Gantt Job Shop Scheduling")
ax.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
random_seed = 2026
import random
random.seed(random_seed)

# Configurable parameters
num_jobs = 10
num_machines = 6
min_time = 1
max_time = 15

# Create machines
machines = [f"M{i+1}" for i in range(num_machines)]

# Create random jobs (each job goes through all machines in random order)
jobs = {}
for j in range(1, num_jobs + 1):
    job_name = f"J{j}"
    machine_sequence = random.sample(machines, len(machines))
    job_ops = [(m, random.randint(min_time, max_time)) for m in machine_sequence]
    jobs[job_name] = job_ops

In [ ]:
operations = [(j,i,m) for j,ops in jobs.items() for i,(m,_) in enumerate(ops)]
durations = {(j,i,m): jobs[j][i][1] for j,i,m in operations}
precedence = [(j,i,m,j,i+1,jobs[j][i+1][0])
              for j,i,m in operations if i+1 < len(jobs[j])]
conflicts = [a+b for idx,a in enumerate(operations) for b in operations[idx+1:] if a[2]==b[2]]
model = new_ampl()
model.eval(r"""
set O dimen 3;
set P dimen 6;
set E dimen 6;
param duration {O} > 0;
param M > 0;
var start {O} >= 0;
var Cmax >= 0;
var before {E} binary;
minimize Total_Cost: Cmax;
subject to Precedence {(j,i,m,j2,i2,m2) in P}:
    start[j2,i2,m2] >= start[j,i,m] + duration[j,i,m];
subject to FirstBeforeSecond {(j,i,m,j2,i2,m2) in E}:
    start[j,i,m] + duration[j,i,m] <= start[j2,i2,m2] + M*(1-before[j,i,m,j2,i2,m2]);
subject to SecondBeforeFirst {(j,i,m,j2,i2,m2) in E}:
    start[j2,i2,m2] + duration[j2,i2,m2] <= start[j,i,m] + M*before[j,i,m,j2,i2,m2];
subject to Completion {(j,i,m) in O}: Cmax >= start[j,i,m] + duration[j,i,m];
""")
model.set["O"] = operations
model.set["P"] = precedence
model.set["E"] = conflicts
model.param["duration"] = durations
model.param["M"] = 1000  # Same bound as the original teaching formulation

solve_checked(model)
start = values(model, "start")
Cmax = model.var["Cmax"].value()
for operation, time in start.items():
    print(f"start{operation} = {time:.1f}")
print(f"Makespan = {Cmax:.1f}")


In [ ]:
# Data for Gantt chart
gantt_data = []
for (j, i, m) in operations:
    s = start[j, i, m]
    d = jobs[j][i][1]
    gantt_data.append((m, s, d, j, i))

# Draw Gantt chart
fig, ax = plt.subplots(figsize=(10, 5))
colors = plt.cm.get_cmap("tab10", num_jobs)
for m, s, d, j, i in gantt_data:
    ax.barh(m, d, left=s, color=colors(int(j[1:]) - 1))
    ax.text(s + d / 2, m, f"{j}-{i}", ha='center', va='center', color='white', fontsize=8)

ax.set_xlabel("Time")
ax.set_ylabel("Machines")
ax.set_title("Job Shop Scheduling Gantt Chart")
# ax.grid(True)
plt.tight_layout()
plt.show()